# Random Forest Regression for Wait Time Prediction

In [ ]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from pyspark.sql.functions import col
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import logging
import warnings

# 1. Initialize MLflow Experiment

# Suppress general Python warnings (like deprecation warnings)
warnings.filterwarnings("ignore")

# Force MLflow's logger to only show errors, hiding all INFO and WARNING logs
logging.getLogger("mlflow").setLevel(logging.ERROR)
# --------------------------------
experiment_name = "PLPA_Dwell_Time_Forecaster_LagFeatures"
mlflow.set_experiment(experiment_name)
mlflow.sklearn.autolog(silent=True)

# 2. Read the Gold Fact Table, filter out closed lanes
spark_df = spark.read.table("Fact_WaitTimes_Gold_New").filter(col("wait_time_min").isNotNull())
df = spark_df.select("snapshot_date", "snapshot_hour", "port_number", "lane_id", "wait_time_min").toPandas()

# 3. Build a proper continuous timestamp so lag/shift operations are unambiguous
df["snapshot_date"] = pd.to_datetime(df["snapshot_date"])
df["timestamp"] = df["snapshot_date"] + pd.to_timedelta(df["snapshot_hour"], unit="h")

# 4. CRITICAL: sort within each port+lane series by actual time.
# Lag features only make sense per-series — you cannot shift across different ports/lanes mixed together.
df = df.sort_values(["port_number", "lane_id", "timestamp"]).reset_index(drop=True)

# 5. Engineer lag and rolling features PER port+lane group
group_cols = ["port_number", "lane_id"]
df["lag_1h"] = df.groupby(group_cols)["wait_time_min"].shift(1)
df["lag_2h"] = df.groupby(group_cols)["wait_time_min"].shift(2)
df["lag_3h"] = df.groupby(group_cols)["wait_time_min"].shift(3)
df["rolling_avg_3h"] = df.groupby(group_cols)["wait_time_min"].transform(
    lambda x: x.shift(1).rolling(3).mean()
)


# Also add day-of-week — cheap to add, directly addresses your "cold start / weekday vs weekend" point
df["day_of_week"] = df["timestamp"].dt.dayofweek  # 0=Monday ... 6=Sunday


# 6. Drop rows where lag features don't exist yet (the first 3 hours of each port+lane series)
# This is expected and correct — you cannot forecast the very first hours of a series with no history.
before_drop = len(df)
df = df.dropna(subset=["lag_1h", "lag_2h", "lag_3h", "rolling_avg_3h"]).reset_index(drop=True)
after_drop = len(df)
print(f"Rows before dropping cold-start NaNs: {before_drop}, after: {after_drop} (dropped {before_drop - after_drop})")

# 7. Re-sort chronologically overall (not by group) for a clean global time split
df = df.sort_values("timestamp").reset_index(drop=True)


# 8. Dynamic Time Split (e.g., 80% Train, 20% Test chronologically)
# This ensures that as your pipeline ingests new daily data, 
# the cutoff shifts automatically without breaking your code.

unique_timestamps = sorted(df["timestamp"].unique())
split_index = int(len(unique_timestamps) * 0.8)
cutoff_timestamp = unique_timestamps[split_index]

train_df = df[df["timestamp"] < cutoff_timestamp].copy()
test_df = df[df["timestamp"] >= cutoff_timestamp].copy()

print(f"Dynamic Cutoff Timestamp: {cutoff_timestamp}")
print(f"Train range: {train_df['timestamp'].min()} to {train_df['timestamp'].max()} ({len(train_df)} rows)")
print(f"Test range: {test_df['timestamp'].min()} to {test_df['timestamp'].max()} ({len(test_df)} rows)")

feature_cols_numeric = ["snapshot_hour", "day_of_week", "lag_1h", "lag_2h", "lag_3h", "rolling_avg_3h"]
categorical_cols = ["port_number", "lane_id"]

X_train_raw = train_df[feature_cols_numeric + categorical_cols]
y_train = train_df["wait_time_min"]
X_test_raw = test_df[feature_cols_numeric + categorical_cols]
y_test = test_df["wait_time_min"]

# 9. One-hot encode categoricals, align test columns to train
X_train = pd.get_dummies(X_train_raw, columns=categorical_cols)
X_test = pd.get_dummies(X_test_raw, columns=categorical_cols)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# 10. Train Random Forest with lag features
with mlflow.start_run() as run:
    print("Training Random Forest Regressor WITH lag features...")
    model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    test_r2 = r2_score(y_test, predictions)

    mlflow.log_metric("test_r2_score", test_r2)

    print(f"Lag-Feature Model — Test MAE: {mae:.2f} minutes")
    print(f"Lag-Feature Model — Test RMSE: {rmse:.2f} minutes")
    print(f"Lag-Feature Model — Test R²: {test_r2:.3f}")

# 11. Honest baseline comparison, computed the SAME way as last time, on the SAME clean split
baseline_lookup = train_df.groupby(["port_number", "lane_id", "snapshot_hour"])["wait_time_min"].mean()

test_df["baseline_pred"] = test_df.apply(
    lambda row: baseline_lookup.get((row["port_number"], row["lane_id"], row["snapshot_hour"]), train_df["wait_time_min"].mean()),
    axis=1
)
baseline_mae = mean_absolute_error(y_test, test_df["baseline_pred"])
baseline_r2 = r2_score(y_test, test_df["baseline_pred"])

print(f"\n--- Comparison on identical clean split ---")
print(f"Naive baseline MAE: {baseline_mae:.2f} | R²: {baseline_r2:.3f}")
print(f"Random Forest (lag features) MAE: {mae:.2f} | R²: {test_r2:.3f}")
print(f"Improvement over baseline: {baseline_mae - mae:.2f} minutes MAE")

# 12. Also add a simpler baseline: "persistence" — predict next hour = last known hour (lag_1h itself)
persistence_mae = mean_absolute_error(y_test, test_df["lag_1h"])
print(f"Persistence baseline (lag_1h as prediction) MAE: {persistence_mae:.2f}")

# 13. Generate predictions across the full dataset and save to Delta
X_all_raw = df[feature_cols_numeric + categorical_cols]
X_all = pd.get_dummies(X_all_raw, columns=categorical_cols)
X_all = X_all.reindex(columns=X_train.columns, fill_value=0)

df["predicted_wait_time_min"] = model.predict(X_all).round(0)

predictions_spark_df = spark.createDataFrame(df.drop(columns=["timestamp"]))
predictions_spark_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("Fact_WaitTime_Predictions")

print("\nPredictions (with lag features) written to Fact_WaitTime_Predictions.")

StatementMeta(, 35e43560-da6e-4019-b849-e586c2e908be, 3, Finished, Available, Finished, False)

Rows before dropping cold-start NaNs: 31377, after: 30715 (dropped 662)
Dynamic Cutoff Timestamp: 2026-07-30 17:00:00
Train range: 2026-07-23 21:00:00 to 2026-07-30 16:00:00 (24451 rows)
Test range: 2026-07-30 17:00:00 to 2026-08-01 09:00:00 (6264 rows)
Training Random Forest Regressor WITH lag features...


Lag-Feature Model — Test MAE: 5.32 minutes
Lag-Feature Model — Test RMSE: 12.24 minutes
Lag-Feature Model — Test R²: 0.795



--- Comparison on identical clean split ---
Naive baseline MAE: 8.99 | R²: 0.587
Random Forest (lag features) MAE: 5.32 | R²: 0.795
Improvement over baseline: 3.67 minutes MAE
Persistence baseline (lag_1h as prediction) MAE: 4.49



Predictions (with lag features) written to Fact_WaitTime_Predictions.


In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

# Isolate: does adding anything on top of lag_1h actually help, or does simpler win?
X_train_lag_only = train_df[["lag_1h"]]
X_test_lag_only = test_df[["lag_1h"]]

model_simple = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
model_simple.fit(X_train_lag_only, y_train)
pred_simple = model_simple.predict(X_test_lag_only)

mae_simple = mean_absolute_error(y_test, pred_simple)
print(f"RF using ONLY lag_1h — MAE: {mae_simple:.2f}")
print(f"Persistence baseline (lag_1h raw) — MAE: {persistence_mae:.2f}")

StatementMeta(, 35e43560-da6e-4019-b849-e586c2e908be, 4, Finished, Available, Finished, False)

RF using ONLY lag_1h — MAE: 5.26
Persistence baseline (lag_1h raw) — MAE: 4.49
